In [19]:
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', None)
data = pd.read_csv('customer_service_faq.csv')
display(data.head())
print('Jumlah baris dan kolom:', data.shape)

,ID,Kategori,Pertanyaan,Jawaban
0,1,Akun & Keamanan,Bagaimana cara mereset password akun saya?,"Anda dapat mereset password melalui menu 'Lupa Kata Sandi' di halaman login. Masukkan email terdaftar, lalu ikuti tautan verifikasi yang dikirimkan ke kotak masuk email Anda."
1,2,Akun & Keamanan,Bagaimana cara mengubah alamat email terdaftar?,"Untuk mengubah alamat email, buka menu Pengaturan Akun > Profil > Ubah Email. Anda perlu memasukkan kode OTP yang dikirimkan ke nomor HP terverifikasi untuk konfirmasi."
2,3,Akun & Keamanan,Bagaimana cara mengganti nomor handphone akun?,"Masuk ke Pengaturan Akun > Keamanan > Nomor Telepon. Pilih 'Ubah Nomor', masukkan nomor HP baru, dan lakukan verifikasi dengan kode OTP yang dikirimkan via SMS."
3,4,Akun & Keamanan,Bagaimana cara mengaktifkan verifikasi dua langkah (2FA)?,Buka Pengaturan Akun > Keamanan > Autentikasi Dua Langkah. Aktifkan opsi OTP via SMS atau aplikasi authenticator (Google Authenticator) untuk mengamankan akun Anda.
4,5,Akun & Keamanan,Bagaimana cara menghapus atau menonaktifkan akun saya?,Anda dapat mengajukan penutupan akun melalui menu Pengaturan > Privasi & Keamanan > Hapus Akun. Pastikan seluruh pesanan dan penarikan saldo sudah selesai sebelum menutup akun.


Jumlah baris dan kolom: (25, 4)


In [20]:
import re
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [stemmer.stem(word) for word in words]
    return ' '.join(words)

data['pertanyaan_clean'] = data['Pertanyaan'].apply(preprocess)
display(data[['Pertanyaan', 'pertanyaan_clean']].head())

,Pertanyaan,pertanyaan_clean
0,Bagaimana cara mereset password akun saya?,mereset password akun
1,Bagaimana cara mengubah alamat email terdaftar?,ubah alamat email daftar
2,Bagaimana cara mengganti nomor handphone akun?,ganti nomor handphone akun
3,Bagaimana cara mengaktifkan verifikasi dua langkah (2FA)?,aktif verifikasi langkah fa
4,Bagaimana cara menghapus atau menonaktifkan akun saya?,hapus nonaktif akun


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(data['pertanyaan_clean'])

print(vectorizer.get_feature_names_out())

['aju' 'aktif' 'akun' 'alamat' 'barang' 'batal' 'bayar' 'biaya' 'butuh'
 'cacat' 'cashback' 'chat' 'cod' 'customer' 'daftar' 'dana' 'diskon'
 'email' 'estimasi' 'fa' 'ganti' 'handphone' 'hapus' 'hasil' 'hubung'
 'jam' 'keluh' 'kembali' 'kirim' 'komplain' 'lacak' 'langgan' 'langkah'
 'layan' 'live' 'mereset' 'mesan' 'metode' 'nomor' 'nonaktif' 'nyata'
 'ongkos' 'operasional' 'otomatis' 'paket' 'password' 'pesan' 'posisi'
 'produk' 'promo' 'prosedur' 'proses' 'refund' 'resi' 'retur' 'rincian'
 'rusak' 'sedia' 'service' 'sistem' 'status' 'support' 'syarat' 'tanggung'
 'tentu' 'terima' 'ubah' 'verifikasi' 'voucher']


In [23]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Ambang batas kemiripan
similarity_threshold = 0.2

# Fungsi chatbot
def chatbot():
    while True:
        query = input("Anda: ")
        
        if query.lower() == 'keluar':
            print("Chatbot: Terima kasih! Sampai jumpa lagi.")
            break

        query_processed = preprocess(query)
        query_tfidf = vectorizer.transform([query_processed])
        
        # Hitung cosine similarity
        similarity_scores = cosine_similarity(query_tfidf, tfidf_matrix)
        top_index = np.argmax(similarity_scores)
        
        # Cek threshold untuk pengalihan ke CS
        if similarity_scores[0, top_index] < similarity_threshold:
            print("Chatbot: Maaf, saya tidak dapat menemukan jawaban yang relevan. Silakan hubungi customer service kami di 0800-123-4567.")
        else:
            print(f"Chatbot: {data['Jawaban'][top_index]}")

print("Chatbot: Halo! Ketik pertanyaan Anda atau ketik 'keluar' untuk mengakhiri obrolan.")
chatbot()

Chatbot: Halo! Ketik pertanyaan Anda atau ketik 'keluar' untuk mengakhiri obrolan.
Chatbot: Anda dapat mereset password melalui menu 'Lupa Kata Sandi' di halaman login. Masukkan email terdaftar, lalu ikuti tautan verifikasi yang dikirimkan ke kotak masuk email Anda.
Chatbot: Anda dapat mengajukan penutupan akun melalui menu Pengaturan > Privasi & Keamanan > Hapus Akun. Pastikan seluruh pesanan dan penarikan saldo sudah selesai sebelum menutup akun.
Chatbot: Maaf, saya tidak dapat menemukan jawaban yang relevan. Silakan hubungi customer service kami di 0800-123-4567.
Chatbot: Terima kasih! Sampai jumpa lagi.
